In [31]:
import pandas as pd
import numpy as np
import os
import json
import shutil
import time
import importlib.util
from datetime import datetime
import anthropic
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
from evaluate import evaluate

# ── CONFIG ────────────────────────────────────────────
API_KEY             = os.environ.get("ANTHROPIC_API_KEY")
MODEL_NAME          = "claude-sonnet-4-6"
MAX_ITERATIONS      = 200                # test run first
LOG_FILE            = "results_log_v4.csv"
STRATEGY_FILE       = "strategy_v4.py"
PROGRAM_EXPLORER    = "program_explorer_v4.txt"
PROGRAM_RISKMANAGER = "program_riskmanager_v4.txt"
MEMORY_FILE         = "shared_memory_v4.json"
HISTORY_WINDOW      = 10
FEE_RATE            = 0.001
MIN_TRADES          = 30
MAX_TRADES          = 456
SHARPE_TOLERANCE    = 0.90

# Risk-aware score parameters
# score = sharpe - penalty for drawdown worse than baseline
# baseline drawdown will be set after first backtest

CHANGE_ROTATION = [
    "Add MA100 trend filter. MA100 = close.rolling(100).mean(). Long only when MA20>MA50 AND MA50>MA100. Short only when MA20<MA50 AND MA50<MA100. Keep the ffill signal logic exactly as in the current code.",
    "Change periods to MA14 and MA30. MA14 = close.rolling(14).mean(). MA30 = close.rolling(30).mean(). Keep exactly the same crossover logic. Do not add any other changes.",
    "Change periods to MA30 and MA100. MA30 = close.rolling(30).mean(). MA100 = close.rolling(100).mean(). Keep exactly the same crossover logic. Do not add any other changes.",
    "Add MA200 filter. MA200 = close.rolling(200).mean(). Long only when MA20>MA50 AND close>MA200. Short only when MA20<MA50 AND close<MA200. Keep the ffill signal logic exactly as in the current code.",
    "Change periods to MA10 and MA40. MA10 = close.rolling(10).mean(). MA40 = close.rolling(40).mean(). Keep exactly the same crossover logic. Do not add any other changes.",
    "Change periods to MA10 and MA40. MA10 = close.rolling(10).mean(). MA40 = close.rolling(40).mean(). Keep exactly the same crossover logic. Do not add any other changes.",
    "Add MA150 filter. MA150 = close.rolling(150).mean(). Long only when MA20>MA50 AND close>MA150. Short only when MA20<MA50 AND close<MA150. Keep the ffill signal logic exactly as in the current code.",
    "Use EMA instead of SMA. EMA20=close.ewm(span=20,adjust=False).mean(). EMA50=close.ewm(span=50,adjust=False).mean(). Long when EMA20>EMA50. Short when EMA20<EMA50. Keep exactly the same signal structure.",
    "Change periods to MA20 and MA100. MA100 = close.rolling(100).mean(). Long when MA20>MA100. Short when MA20<MA100. No other changes.",
    "Add MA75 filter. MA75 = close.rolling(75).mean(). Long only when MA20>MA50 AND MA50>MA75. Short only when MA20<MA50 AND MA50<MA75. Keep the ffill signal logic exactly as in the current code."
]
# ──────────────────────────────────────────────────────

client = anthropic.Anthropic(api_key=API_KEY)

test = client.messages.create(
    model=MODEL_NAME,
    max_tokens=10,
    messages=[{"role": "user", "content": "say only the word: connected"}]
)
print("Claude status:", test.content[0].text.strip())

Claude status: connected


In [33]:
df_check = pd.read_csv('data/btc_hourly.csv', parse_dates=['date'])
val_check = df_check[(df_check['date'] >= '2023-01-01') &
                     (df_check['date'] < '2024-01-01')].copy()

spec = importlib.util.spec_from_file_location("strategy", STRATEGY_FILE)
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)

result_check = mod.run_strategy(val_check)
metrics_check = evaluate(result_check)
trades_check = int((result_check['signal'].diff().abs() > 0).sum())

print(f"Baseline Sharpe:  {metrics_check['sharpe']}")
print(f"Baseline Trades:  {trades_check}")

if abs(metrics_check['sharpe'] - 0.5646) < 0.01:
    print("Correct baseline confirmed")
else:
    print("Wrong — check strategy_v4.py in Spyder")

Baseline Sharpe:  0.5646
Baseline Trades:  228
Correct baseline confirmed


In [35]:
def compute_score(sharpe, max_drawdown, baseline_drawdown):
    """
    Risk-aware score:
    Start with Sharpe, subtract penalty if drawdown is worse than baseline.
    Penalty = how much worse drawdown is as a fraction of baseline drawdown.
    If drawdown improved or stayed same, no penalty.
    """
    baseline_dd_abs = abs(baseline_drawdown)
    strategy_dd_abs = abs(max_drawdown)

    if strategy_dd_abs <= baseline_dd_abs:
        penalty = 0
    else:
        penalty = (strategy_dd_abs - baseline_dd_abs) / baseline_dd_abs

    score = sharpe - penalty
    return round(score, 4)

# Test the score function
print("Score examples:")
print(f"  Same drawdown as baseline, Sharpe 1.50: {compute_score(1.50, -45.85, -45.85)}")
print(f"  Better drawdown, Sharpe 1.40:           {compute_score(1.40, -30.00, -45.85)}")
print(f"  Worse drawdown by 20%, Sharpe 1.60:     {compute_score(1.60, -55.00, -45.85)}")
print(f"  Much worse drawdown, Sharpe 1.80:       {compute_score(1.80, -70.00, -45.85)}")

Score examples:
  Same drawdown as baseline, Sharpe 1.50: 1.5
  Better drawdown, Sharpe 1.40:           1.4
  Worse drawdown by 20%, Sharpe 1.60:     1.4004
  Much worse drawdown, Sharpe 1.80:       1.2733


In [37]:
def run_backtest():
    df = pd.read_csv('data/btc_hourly.csv', parse_dates=['date'])
    val = df[(df['date'] >= '2023-01-01') &
             (df['date'] < '2024-01-01')].copy()

    spec = importlib.util.spec_from_file_location("strategy", STRATEGY_FILE)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)

    result = mod.run_strategy(val)

    if "strategy_returns" not in result.columns:
        raise ValueError("strategy_returns column missing")

    trades = result["signal"].diff().abs() > 0
    n_trades = int(trades.sum())
    turnover_per_day = round(n_trades / (len(result) / 24), 2)

    metrics = evaluate(result)
    metrics["n_trades"] = n_trades
    metrics["turnover_per_day"] = turnover_per_day

    return metrics

print("run_backtest() ready")

run_backtest() ready


In [39]:
def validate_code(code):
    forbidden = [
        "import scipy", "import sklearn", "import torch",
        "import tensorflow", "import statsmodels", "import talib",
        "import requests", "import urllib", "import warnings"
    ]
    for lib in forbidden:
        if lib in code:
            return False, f"forbidden import: {lib}"

    if "def evaluate(" in code:
        return False, "agent redefined evaluate()"

    if "strategy_returns" not in code:
        return False, "strategy_returns column missing"

    if not code.strip().startswith("import pandas"):
        return False, "code does not start with import pandas"

    try:
        compile(code, "<string>", "exec")
    except SyntaxError as e:
        return False, f"syntax error: {e}"

    return True, None

print("validate_code() ready")

validate_code() ready


In [41]:
def build_history(n=HISTORY_WINDOW):
    if not os.path.exists(LOG_FILE):
        return "No experiments yet."
    log = pd.read_csv(LOG_FILE)
    if len(log) == 0:
        return "No experiments yet."
    recent = log.tail(n)
    lines = []
    for _, row in recent.iterrows():
        status = "KEPT" if row['kept'] else "REVERTED"
        lines.append(
            f"Iter {int(row['iteration'])}: "
            f"Sharpe={row['sharpe']} | "
            f"Drawdown={row['max_drawdown']}% | "
            f"Score={row['score']} | "
            f"Trades={row['n_trades']} | "
            f"Agent={row['agent']} | "
            f"{status} | {row['note']}"
        )
    return "\n".join(lines)


def build_agent2_history(n=5):
    if not os.path.exists(LOG_FILE):
        return "No Risk Manager experiments yet."
    log = pd.read_csv(LOG_FILE)
    a2 = log[log['agent'] == 'agent2']
    if len(a2) == 0:
        return "No Risk Manager experiments yet."
    recent = a2.tail(n)
    lines = []
    for _, row in recent.iterrows():
        status = "KEPT" if row['kept'] else "REVERTED"
        lines.append(
            f"Iter {int(row['iteration'])}: "
            f"Sharpe={row['sharpe']} | "
            f"Drawdown={row['max_drawdown']}% | "
            f"Score={row['score']} | "
            f"{status} | {row['note']}"
        )
    return "\n".join(lines)


def build_forbidden_sharpes():
    if not os.path.exists(LOG_FILE):
        return ""
    log = pd.read_csv(LOG_FILE)
    a1 = log[log['agent'] == 'agent1']
    if len(a1) == 0:
        return ""
    counts = a1['sharpe'].value_counts()
    repeated = counts[counts >= 2].index.tolist()
    if not repeated:
        return ""
    lines = "\nFORBIDDEN Sharpe values — never propose these again:\n"
    for s in repeated:
        lines += f"{float(s):.4f} is FORBIDDEN\n"
    return lines

print("History builders ready")

History builders ready


In [43]:
def read_memory():
    if not os.path.exists(MEMORY_FILE):
        return {
            "iteration": 0,
            "risk_level": "unknown",
            "main_weakness": "No feedback yet. First iteration.",
            "recommendation": "Explore freely. Try adding MA100 as trend confirmation.",
            "modification_needed": False,
            "modified_code": ""
        }
    with open(MEMORY_FILE, 'r', encoding='utf-8') as f:
        return json.load(f)

def write_memory(data):
    with open(MEMORY_FILE, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=2)

print("Shared memory functions ready")

Shared memory functions ready


In [45]:
def ask_explorer(current_code, current_sharpe, best_drawdown,
                 baseline_drawdown, iteration):

    with open(PROGRAM_EXPLORER, 'r', encoding='utf-8') as f:
        prompt = f.read()

    memory = read_memory()
    forced_change = CHANGE_ROTATION[iteration % len(CHANGE_ROTATION)]
    history_text = build_history() + build_forbidden_sharpes()

    prompt = prompt.replace('{sharpe}', str(current_sharpe))
    prompt = prompt.replace('{best_drawdown}', str(best_drawdown))
    prompt = prompt.replace('{baseline_drawdown}', str(baseline_drawdown))
    prompt = prompt.replace('{forced_change}', forced_change)
    prompt = prompt.replace('{risk_level}', memory['risk_level'])
    prompt = prompt.replace('{main_weakness}', memory['main_weakness'])
    prompt = prompt.replace('{recommendation}', memory['recommendation'])
    prompt = prompt.replace('{history}', history_text)
    prompt = prompt.replace('{code}', current_code)

    response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=4096,
        messages=[{"role": "user", "content": prompt}]
    )
    new_code = response.content[0].text.strip()

    # Strip markdown fences
    if new_code.startswith("```"):
        lines = new_code.split('\n')
        lines = [l for l in lines if not l.startswith("```")]
        new_code = '\n'.join(lines).strip()

    # Strip preamble
    if "import pandas" in new_code:
        idx = new_code.index("import pandas")
        new_code = new_code[idx:]

    time.sleep(3)
    return new_code

print("ask_explorer() ready")

ask_explorer() ready


In [47]:
def ask_riskmanager(current_code, metrics_a1, best_sharpe,
                    best_drawdown, baseline_drawdown, best_score):

    with open(PROGRAM_RISKMANAGER, 'r', encoding='utf-8') as f:
        prompt = f.read()

    score_threshold = round(best_score * 0.95, 4)

    prompt = prompt.replace('{sharpe}', str(metrics_a1['sharpe']))
    prompt = prompt.replace('{drawdown}', str(metrics_a1['max_drawdown']))
    prompt = prompt.replace('{n_trades}', str(metrics_a1['n_trades']))
    prompt = prompt.replace('{calmar}', str(metrics_a1['calmar']))
    prompt = prompt.replace('{best_sharpe}', str(best_sharpe))
    prompt = prompt.replace('{best_drawdown}', str(best_drawdown))
    prompt = prompt.replace('{baseline_drawdown}', str(baseline_drawdown))
    prompt = prompt.replace('{score_threshold}', str(score_threshold))
    prompt = prompt.replace('{agent2_history}', build_agent2_history())
    prompt = prompt.replace('{code}', current_code)

    response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=4096,
        messages=[{"role": "user", "content": prompt}]
    )

    raw = response.content[0].text.strip()
    time.sleep(3)
    return raw

print("ask_riskmanager() ready")

ask_riskmanager() ready


In [49]:
def parse_riskmanager_response(raw):
    """
    Parse Agent 2's JSON response.
    Returns a dict with all 5 fields.
    Falls back to safe defaults if parsing fails.
    """
    # Strip markdown fences if present
    if "```" in raw:
        lines = raw.split('\n')
        lines = [l for l in lines if not l.startswith("```")]
        raw = '\n'.join(lines).strip()

    # Find JSON object
    start = raw.find('{')
    end = raw.rfind('}') + 1
    if start != -1 and end > start:
        raw = raw[start:end]

    try:
        data = json.loads(raw)

        # Ensure all required fields exist
        result = {
            "risk_level": str(data.get("risk_level", "medium")),
            "main_weakness": str(data.get("main_weakness", "unknown")),
            "recommendation": str(data.get("recommendation", "try a different approach")),
            "modification_needed": bool(data.get("modification_needed", False)),
            "modified_code": str(data.get("modified_code", ""))
        }

        # Clean modified code if present
        if result["modified_code"]:
            code = result["modified_code"]
            if code.startswith("```"):
                lines = code.split('\n')
                lines = [l for l in lines if not l.startswith("```")]
                code = '\n'.join(lines).strip()
            if "import pandas" in code:
                idx = code.index("import pandas")
                code = code[idx:]
            result["modified_code"] = code

        return result

    except json.JSONDecodeError as e:
        print(f"  JSON parse error: {e}")
        print(f"  Raw response: {raw[:200]}")
        return {
            "risk_level": "medium",
            "main_weakness": "Could not parse response",
            "recommendation": "Try a different signal approach",
            "modification_needed": False,
            "modified_code": ""
        }

print("parse_riskmanager_response() ready")

parse_riskmanager_response() ready


In [51]:
def log_result(iteration, metrics, kept, note, agent,
               score=None, code=None):
    row = {
        'iteration':         iteration,
        'timestamp':         datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'agent':             agent,
        'sharpe':            metrics.get('sharpe', None),
        'score':             score,
        'calmar':            metrics.get('calmar', None),
        'total_return':      metrics.get('total_return', None),
        'max_drawdown':      metrics.get('max_drawdown', None),
        'n_trades':          metrics.get('n_trades', None),
        'turnover_per_day':  metrics.get('turnover_per_day', None),
        'kept':              kept,
        'note':              note
    }

    df_log = pd.DataFrame([row])
    write_header = not os.path.exists(LOG_FILE)
    df_log.to_csv(LOG_FILE, mode='a', header=write_header, index=False)

    if kept and code:
        folder = "kept_strategies_v4"
        os.makedirs(folder, exist_ok=True)
        fname = (
            f"{folder}/strategy_{agent}_"
            f"iter{iteration}_"
            f"sharpe{metrics['sharpe']}_"
            f"score{score}.py"
        )
        with open(fname, 'w') as f:
            f.write(code)

print("log_result() ready")

log_result() ready


In [53]:
# ── CRASH RECOVERY OR FRESH START ─────────────────────
if os.path.exists(LOG_FILE):
    existing_log = pd.read_csv(LOG_FILE)
    completed = len(existing_log[existing_log['agent'] == 'agent1'])
    kept_rows = existing_log[existing_log['kept'] == True]
    best_sharpe   = kept_rows['sharpe'].max()
    best_drawdown = kept_rows.loc[kept_rows['sharpe'].idxmax(), 'max_drawdown']
    best_calmar   = kept_rows.loc[kept_rows['sharpe'].idxmax(), 'calmar']
    best_score    = kept_rows['score'].max()
    baseline_row  = existing_log[existing_log['agent'] == 'baseline'].iloc[0]
    baseline_sharpe   = baseline_row['sharpe']
    baseline_drawdown = baseline_row['max_drawdown']

    best_iter = int(kept_rows.loc[kept_rows['score'].idxmax(), 'iteration'])
    if os.path.exists('kept_strategies_v4'):
        kept_files = os.listdir('kept_strategies_v4')
        best_files = [f for f in kept_files if f'iter{best_iter}_' in f]
        if best_files:
            with open(f'kept_strategies_v4/{best_files[0]}', 'r') as f:
                best_code = f.read()
            with open(STRATEGY_FILE, 'w') as f:
                f.write(best_code)
        else:
            with open(STRATEGY_FILE, 'r') as f:
                best_code = f.read()
    else:
        with open(STRATEGY_FILE, 'r') as f:
            best_code = f.read()

    START_ITERATION = completed + 1
    print(f"Resuming from iteration {START_ITERATION}")
    print(f"Best Sharpe so far:    {best_sharpe}")
    print(f"Best Score so far:     {best_score}")
    print(f"Completed so far:      {completed} iterations")

else:
    # Fresh start
    print("Running baseline backtest...")
    try:
        baseline = run_backtest()
    except Exception as e:
        print(f"Baseline failed: {e}")
        raise

    baseline_sharpe   = baseline['sharpe']
    baseline_drawdown = baseline['max_drawdown']
    baseline_score    = compute_score(
        baseline['sharpe'], baseline['max_drawdown'], baseline['max_drawdown']
    )

    best_sharpe   = baseline_sharpe
    best_drawdown = baseline_drawdown
    best_calmar   = baseline['calmar']
    best_score    = baseline_score

    print(f"\n=== Baseline Results ===")
    print(f"  Sharpe:       {baseline_sharpe}")
    print(f"  Drawdown:     {baseline_drawdown}%")
    print(f"  Score:        {baseline_score}")
    print(f"  Trades:       {baseline['n_trades']}")

    with open(STRATEGY_FILE, 'r') as f:
        best_code = f.read()

    log_result(0, baseline, True, "baseline", "baseline",
               score=baseline_score, code=best_code)
    START_ITERATION = 1

# Reset shared memory for fresh start
if START_ITERATION == 1:
    write_memory({
        "iteration": 0,
        "risk_level": "unknown",
        "main_weakness": "No feedback yet. First iteration.",
        "recommendation": "Explore freely. Try adding MA100 as trend confirmation.",
        "modification_needed": False,
        "modified_code": ""
    })

# ── MAIN LOOP ──────────────────────────────────────────
print(f"\nStarting V4 two-agent loop — {MAX_ITERATIONS} iterations")
print("=" * 55)

for i in range(START_ITERATION, MAX_ITERATIONS + 1):
    print(f"\n[Iteration {i}/{MAX_ITERATIONS}]")
    print(f"  Forced change: {CHANGE_ROTATION[i % len(CHANGE_ROTATION)][:55]}...")

    # ── EXPLORER ──────────────────────────────────────
    print("  [Explorer] Proposing...")
    try:
        new_code_a1 = ask_explorer(
            best_code, best_sharpe, best_drawdown,
            baseline_drawdown, i
        )
    except Exception as e:
        print(f"  Explorer error: {e}")
        log_result(i, {'sharpe': best_sharpe, 'calmar': best_calmar,
                       'total_return': None, 'max_drawdown': best_drawdown,
                       'n_trades': None, 'turnover_per_day': None},
                   False, f"explorer error: {e}", "agent1",
                   score=None)
        time.sleep(15)
        continue

    # Code guard
    valid, reason = validate_code(new_code_a1)
    if not valid:
        print(f"  Explorer REJECTED by guard: {reason}")
        log_result(i, {'sharpe': best_sharpe, 'calmar': best_calmar,
                       'total_return': None, 'max_drawdown': best_drawdown,
                       'n_trades': None, 'turnover_per_day': None},
                   False, f"guard: {reason}", "agent1", score=None)
        continue

    # Write and backtest
    shutil.copy(STRATEGY_FILE, STRATEGY_FILE + '.backup')
    with open(STRATEGY_FILE, 'w') as f:
        f.write(new_code_a1)

    print("  [Explorer] Running backtest...")
    try:
        metrics_a1 = run_backtest()
        score_a1 = compute_score(
            metrics_a1['sharpe'],
            metrics_a1['max_drawdown'],
            baseline_drawdown
        )
        print(f"  [Explorer] Sharpe={metrics_a1['sharpe']} | "
              f"Drawdown={metrics_a1['max_drawdown']}% | "
              f"Score={score_a1} | "
              f"Trades={metrics_a1['n_trades']}")
    except Exception as e:
        print(f"  [Explorer] Backtest error: {e}")
        shutil.copy(STRATEGY_FILE + '.backup', STRATEGY_FILE)
        log_result(i, {'sharpe': best_sharpe, 'calmar': best_calmar,
                       'total_return': None, 'max_drawdown': best_drawdown,
                       'n_trades': None, 'turnover_per_day': None},
                   False, f"backtest error: {e}", "agent1", score=None)
        continue

    # Trade guards
    if metrics_a1['n_trades'] < MIN_TRADES:
        print(f"  Explorer REJECTED — too few trades ({metrics_a1['n_trades']})")
        shutil.copy(STRATEGY_FILE + '.backup', STRATEGY_FILE)
        log_result(i, metrics_a1, False,
                   f"too few trades: {metrics_a1['n_trades']}",
                   "agent1", score=score_a1)
        continue

    if metrics_a1['n_trades'] > MAX_TRADES:
        print(f"  Explorer REJECTED — too many trades ({metrics_a1['n_trades']})")
        shutil.copy(STRATEGY_FILE + '.backup', STRATEGY_FILE)
        log_result(i, metrics_a1, False,
                   f"too many trades: {metrics_a1['n_trades']}",
                   "agent1", score=score_a1)
        continue

    # ── RISK MANAGER (always runs) ─────────────────────
    print("  [Risk Manager] Reviewing...")
    try:
        raw_response = ask_riskmanager(
            new_code_a1, metrics_a1,
            best_sharpe, best_drawdown,
            baseline_drawdown, best_score
        )
        memory_update = parse_riskmanager_response(raw_response)
        print(f"  [Risk Manager] Risk={memory_update['risk_level']} | "
              f"Weakness: {memory_update['main_weakness'][:60]}...")
        print(f"  [Risk Manager] Modification needed: {memory_update['modification_needed']}")
    except Exception as e:
        print(f"  Risk Manager error: {e}")
        memory_update = {
            "risk_level": "medium",
            "main_weakness": f"Error: {e}",
            "recommendation": "Try a different signal approach",
            "modification_needed": False,
            "modified_code": ""
        }

    # Save feedback to shared memory
    memory_update["iteration"] = i
    write_memory(memory_update)

    # ── RISK MANAGER MODIFICATION ──────────────────────
    score_a2 = None
    metrics_a2 = None
    modified_code = memory_update["modified_code"]

    if memory_update["modification_needed"] and modified_code:
        print("  [Risk Manager] Testing risk overlay...")

        valid, reason = validate_code(modified_code)
        if not valid:
            print(f"  Risk Manager code REJECTED: {reason}")
            log_result(i, {'sharpe': metrics_a1['sharpe'],
                           'calmar': metrics_a1['calmar'],
                           'total_return': None,
                           'max_drawdown': metrics_a1['max_drawdown'],
                           'n_trades': None, 'turnover_per_day': None},
                       False, f"agent2 guard: {reason}",
                       "agent2", score=None)
        else:
            shutil.copy(STRATEGY_FILE, STRATEGY_FILE + '.a2backup')
            with open(STRATEGY_FILE, 'w') as f:
                f.write(modified_code)

            try:
                metrics_a2 = run_backtest()
                score_a2 = compute_score(
                    metrics_a2['sharpe'],
                    metrics_a2['max_drawdown'],
                    baseline_drawdown
                )
                print(f"  [Risk Manager] Sharpe={metrics_a2['sharpe']} | "
                      f"Drawdown={metrics_a2['max_drawdown']}% | "
                      f"Score={score_a2} | "
                      f"Trades={metrics_a2['n_trades']}")
            except Exception as e:
                print(f"  Risk Manager backtest error: {e}")
                shutil.copy(STRATEGY_FILE + '.a2backup', STRATEGY_FILE)
                metrics_a2 = None
                score_a2 = None
                log_result(i, {'sharpe': metrics_a1['sharpe'],
                               'calmar': metrics_a1['calmar'],
                               'total_return': None,
                               'max_drawdown': metrics_a1['max_drawdown'],
                               'n_trades': None, 'turnover_per_day': None},
                           False, f"agent2 backtest error: {e}",
                           "agent2", score=None)

    # ── CONTROLLER DECISION ────────────────────────────
    print(f"\n  [Controller] Comparing candidates...")
    print(f"  Current best score: {best_score}")
    print(f"  Explorer score:     {score_a1}")
    print(f"  Risk Manager score: {score_a2}")

    # Find best candidate
    candidates = [
        ("current_best", best_score, best_code,
         {'sharpe': best_sharpe, 'max_drawdown': best_drawdown,
          'calmar': best_calmar, 'n_trades': None,
          'total_return': None, 'turnover_per_day': None}),
        ("agent1", score_a1, new_code_a1, metrics_a1),
    ]

    if score_a2 is not None and metrics_a2 is not None:
        candidates.append(("agent2", score_a2, modified_code, metrics_a2))

    # Sort by score descending
    candidates.sort(key=lambda x: x[1], reverse=True)
    winner_name, winner_score, winner_code, winner_metrics = candidates[0]

    if winner_name == "current_best":
        # No improvement — revert
        shutil.copy(STRATEGY_FILE + '.backup', STRATEGY_FILE)
        print(f"  Controller: REVERT — current best wins")

        if score_a2 is not None:
            log_result(i, metrics_a2, False,
                       "riskmanager no improvement", "agent2",
                       score=score_a2)
        log_result(i, metrics_a1, False,
                   "explorer no improvement", "agent1",
                   score=score_a1)

    elif winner_name == "agent1":
        # Explorer wins
        with open(STRATEGY_FILE, 'w') as f:
            f.write(winner_code)
        best_sharpe   = winner_metrics['sharpe']
        best_drawdown = winner_metrics['max_drawdown']
        best_calmar   = winner_metrics['calmar']
        best_score    = winner_score
        best_code     = winner_code
        print(f"  Controller: KEEP Explorer — "
              f"Score={winner_score} Sharpe={winner_metrics['sharpe']}")

        if score_a2 is not None:
            log_result(i, metrics_a2, False,
                       "riskmanager lower score than explorer",
                       "agent2", score=score_a2)
        log_result(i, winner_metrics, True,
                   "explorer improved score", "agent1",
                   score=winner_score, code=winner_code)

    elif winner_name == "agent2":
        # Risk Manager wins
        with open(STRATEGY_FILE, 'w') as f:
            f.write(winner_code)
        best_sharpe   = winner_metrics['sharpe']
        best_drawdown = winner_metrics['max_drawdown']
        best_calmar   = winner_metrics['calmar']
        best_score    = winner_score
        best_code     = winner_code
        print(f"  Controller: KEEP Risk Manager — "
              f"Score={winner_score} Sharpe={winner_metrics['sharpe']} "
              f"Drawdown={winner_metrics['max_drawdown']}%")

        log_result(i, metrics_a1, False,
                   "explorer lower score than riskmanager",
                   "agent1", score=score_a1)
        log_result(i, winner_metrics, True,
                   "riskmanager improved score", "agent2",
                   score=winner_score, code=winner_code)

# ── SUMMARY ───────────────────────────────────────────
print("\n" + "=" * 55)
print(f"V4 two-agent loop finished!")
print(f"Best Sharpe achieved:   {best_sharpe}")
print(f"Best Score achieved:    {best_score}")
print(f"Best Drawdown achieved: {best_drawdown}%")
print(f"Results saved to:       {LOG_FILE}")
print(f"Kept strategies in:     kept_strategies_v4/")

Running baseline backtest...

=== Baseline Results ===
  Sharpe:       0.5646
  Drawdown:     -45.85%
  Score:        0.5646
  Trades:       228

Starting V4 two-agent loop — 200 iterations

[Iteration 1/200]
  Forced change: Change periods to MA14 and MA30. MA14 = close.rolling(1...
  [Explorer] Proposing...
  [Explorer] Running backtest...
  [Explorer] Sharpe=0.0177 | Drawdown=-58.02% | Score=-0.2477 | Trades=344
  [Risk Manager] Reviewing...
  [Risk Manager] Risk=high | Weakness: Strategy has a catastrophic drawdown of -58.02% which is sig...
  [Risk Manager] Modification needed: False

  [Controller] Comparing candidates...
  Current best score: 0.5646
  Explorer score:     -0.2477
  Risk Manager score: None
  Controller: REVERT — current best wins

[Iteration 2/200]
  Forced change: Change periods to MA30 and MA100. MA30 = close.rolling(...
  [Explorer] Proposing...
  [Explorer] Running backtest...
  [Explorer] Sharpe=0.3211 | Drawdown=-38.32% | Score=0.3211 | Trades=117
  [Risk M

In [55]:
log = pd.read_csv(LOG_FILE)

a1_log = log[log['agent'] == 'agent1']
a2_log = log[log['agent'] == 'agent2']
baseline_log = log[log['agent'] == 'baseline']

print("=== V4 TWO-AGENT EXPERIMENT SUMMARY ===")
print(f"\nBaseline Sharpe:         {baseline_log['sharpe'].values[0]}")
print(f"Baseline Score:          {baseline_log['score'].values[0]}")
print(f"Best Sharpe achieved:    {log['sharpe'].max()}")
print(f"Best Score achieved:     {log['score'].max()}")

print(f"\nExplorer Agent (Agent 1):")
print(f"  Total proposals:       {len(a1_log)}")
print(f"  Kept:                  {a1_log['kept'].sum()}")
print(f"  Too many trades:       {a1_log['note'].str.contains('too many').sum()}")
print(f"  Guard rejections:      {a1_log['note'].str.contains('guard').sum()}")
print(f"  Best Sharpe:           {a1_log['sharpe'].max()}")
print(f"  Best Score:            {a1_log['score'].max()}")

print(f"\nRisk Manager Agent (Agent 2):")
print(f"  Total reviews:         {len(a2_log)}")
print(f"  Kept:                  {a2_log['kept'].sum()}")
print(f"  Best Score:            {a2_log['score'].max() if len(a2_log) > 0 else 'N/A'}")

print(f"\nFull log:")
print(log.to_string())

=== V4 TWO-AGENT EXPERIMENT SUMMARY ===

Baseline Sharpe:         0.5646
Baseline Score:          0.5646
Best Sharpe achieved:    2.0216
Best Score achieved:     2.0216

Explorer Agent (Agent 1):
  Total proposals:       200
  Kept:                  5
  Too many trades:       0
  Guard rejections:      0
  Best Sharpe:           2.0042
  Best Score:            2.0042

Risk Manager Agent (Agent 2):
  Total reviews:         7
  Kept:                  2
  Best Score:            2.0216

Full log:
     iteration            timestamp     agent  sharpe   score  calmar  total_return  max_drawdown  n_trades  turnover_per_day   kept                                   note
0            0  2026-05-19 13:35:39  baseline  0.5646  0.5646  0.3510         16.09        -45.85       228              0.63   True                               baseline
1            1  2026-05-19 13:35:54    agent1  0.0177 -0.2477 -0.1370         -7.95        -58.02       344              0.95  False                explorer n